In [4]:
# DEFINING THE FILE PATH AND READING THE HEADER OF THE FILE 
import pandas as pd
df = pd.read_csv("/projectnb/bf528/students/shrjain/final-project-template-shriyajain22/results/diffbind_csv/cDC1_WT_vs_KO_diffpeaks_ALL.csv")
print(df.columns.tolist())

['seqnames', 'start', 'end', 'width', 'strand', 'Conc', 'Conc_WT', 'Conc_KO', 'Fold', 'p.value', 'FDR']


In [5]:
# CONVERTING THE CSV FILE TO BED FILE 
import pandas as pd

def csv_to_bed(in_csv, out_bed):
    df = pd.read_csv(in_csv)

    colmap = {
        "Chr": ["Chr", "chr", "Chromosome", "seqnames"],
        "Start": ["Start", "start"],
        "End": ["End", "end"]
    }

    coords = {}
    for key, options in colmap.items():
        for opt in options:
            if opt in df.columns:
                coords[key] = opt
                break

    if len(coords) != 3:
        raise ValueError(f"Could not detect coordinate columns. Found: {df.columns.tolist()}")

    # DETECTING THE FOLD CHANGE AND FDR 
    if "Fold" in df.columns:
        lfc_col = "Fold"
    elif "log2FoldChange" in df.columns:
        lfc_col = "log2FoldChange"
    else:
        raise ValueError("Could not find Fold or log2FoldChange column")

    if "FDR" in df.columns:
        fdr_col = "FDR"
    elif "padj" in df.columns:
        fdr_col = "padj"
    else:
        raise ValueError("Could not find FDR or padj column")

    bed = df[[coords["Chr"], coords["Start"], coords["End"], lfc_col, fdr_col]].copy()
    bed["peak_id"] = [f"peak_{i}" for i in range(len(bed))]

    bed = bed[[coords["Chr"], coords["Start"], coords["End"], "peak_id", lfc_col, fdr_col]]
    bed.to_csv(out_bed, sep="\t", header=False, index=False)
    print("Wrote", out_bed)

In [7]:
csv_to_bed(
    "/projectnb/bf528/students/shrjain/final-project-template-shriyajain22/results/diffbind_csv/cDC1_WT_vs_KO_diffpeaks_ALL.csv",
    "cDC1_diffpeaks.bed"
)

csv_to_bed(
    "/projectnb/bf528/students/shrjain/final-project-template-shriyajain22/results/diffbind_csv/cDC2_WT_vs_KO_diffpeaks_ALL.csv",
    "cDC2_diffpeaks.bed"
)


Wrote cDC1_diffpeaks.bed
Wrote cDC2_diffpeaks.bed


In [8]:
# IMPORTING THE REQUIRED LIBRARIES 
import gzip
import pandas as pd
import numpy as np
import os

In [9]:
# DEFINING THE PATHS FOR REQUIRED FILES 
GTF_GZ = "/projectnb/bf528/students/shrjain/final-project-template-shriyajain22/refs/gencode.vM38.primary_assembly.annotation.gtf.gz"

CDC1_DIFFBIND = "/projectnb/bf528/students/shrjain/final-project-template-shriyajain22/results/diffbind_csv/cDC1_WT_vs_KO_diffpeaks_ALL.csv"
CDC2_DIFFBIND = "/projectnb/bf528/students/shrjain/final-project-template-shriyajain22/results/diffbind_csv/cDC2_WT_vs_KO_diffpeaks_ALL.csv"

OUT_CDC1 = "/projectnb/bf528/students/shrjain/final-project-template-shriyajain22/refs/cdc1_atac_DAR_with_genes.tsv"
OUT_CDC2 = "/projectnb/bf528/students/shrjain/final-project-template-shriyajain22/refs/cdc2_atac_DAR_with_genes.tsv"

In [10]:
def parse_gene_tss_from_gtf(gtf_gz_path):
    rows = []
    with gzip.open(gtf_gz_path, "rt") as f:
        for line in f:
            if line.startswith("#"):
                continue
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 9:
                continue
            chrom, source, feature, start, end, score, strand, frame, attrs = parts
            if feature != "gene":
                continue
            start = int(start)
            end   = int(end)

            gene_id = None
            for field in attrs.split(";"):
                field = field.strip()
                if field.startswith("gene_id"):
                    gene_id = field.split(" ")[1].replace('"', "")
                    break
            if gene_id is None:
                continue

            tss = start if strand == "+" else end
            rows.append((chrom, tss, gene_id))

    tss_df = pd.DataFrame(rows, columns=["chr", "tss", "gene"])
    tss_df = tss_df.sort_values(["chr", "tss"]).reset_index(drop=True)
    return tss_df

def detect_diffbind_cols(df):
    chr_col = next((c for c in ["Chr","chr","Chromosome","seqnames"] if c in df.columns), None)
    start_col = next((c for c in ["Start","start"] if c in df.columns), None)
    end_col = next((c for c in ["End","end"] if c in df.columns), None)

    lfc_col = "Fold" if "Fold" in df.columns else ("log2FoldChange" if "log2FoldChange" in df.columns else None)
    fdr_col = "FDR" if "FDR" in df.columns else ("padj" if "padj" in df.columns else None)

    if None in [chr_col, start_col, end_col, lfc_col, fdr_col]:
        raise ValueError(f"Could not detect required columns.\nFound columns:\n{df.columns.tolist()}")

    return chr_col, start_col, end_col, lfc_col, fdr_col

def map_peaks_to_nearest_gene(peaks_df, tss_df):
    tss_by_chr = {}
    gene_by_chr = {}
    for chrom, sub in tss_df.groupby("chr", sort=False):
        tss_by_chr[chrom] = sub["tss"].to_numpy(dtype=int)
        gene_by_chr[chrom] = sub["gene"].to_numpy(dtype=str)

    nearest_genes = []
    nearest_dist = []

    for chrom, center in zip(peaks_df["chr"], peaks_df["center"]):
        if chrom not in tss_by_chr:
            nearest_genes.append(np.nan)
            nearest_dist.append(np.nan)
            continue

        arr = tss_by_chr[chrom]
        genes = gene_by_chr[chrom]

        idx = np.searchsorted(arr, center)

        best_gene = None
        best_dist = None

        if idx > 0:
            d = abs(center - arr[idx-1])
            best_gene = genes[idx-1]
            best_dist = d
        if idx < len(arr):
            d = abs(center - arr[idx])
            if best_dist is None or d < best_dist:
                best_gene = genes[idx]
                best_dist = d

        nearest_genes.append(best_gene)
        nearest_dist.append(best_dist)

    peaks_df = peaks_df.copy()
    peaks_df["gene"] = nearest_genes
    peaks_df["dist_to_tss"] = nearest_dist
    return peaks_df

def build_atac_gene_table(diffbind_csv, out_tsv, tss_df):
    df = pd.read_csv(diffbind_csv)
    chr_col, start_col, end_col, lfc_col, fdr_col = detect_diffbind_cols(df)

    peaks = df[[chr_col, start_col, end_col, lfc_col, fdr_col]].copy()
    peaks.columns = ["chr","start","end","atac_log2fc","atac_fdr"]
    peaks["start"] = peaks["start"].astype(int)
    peaks["end"]   = peaks["end"].astype(int)
    peaks["center"] = ((peaks["start"] + peaks["end"]) // 2).astype(int)

    mapped = map_peaks_to_nearest_gene(peaks, tss_df)
    mapped = mapped.dropna(subset=["gene"]).copy()

    mapped["abs_lfc"] = mapped["atac_log2fc"].abs()
    gene_df = mapped.sort_values("abs_lfc", ascending=False).drop_duplicates("gene")

    gene_df = gene_df[["gene","atac_log2fc","atac_fdr"]]
    os.makedirs(os.path.dirname(out_tsv), exist_ok=True)
    gene_df.to_csv(out_tsv, sep="\t", index=False)
    print("Wrote:", out_tsv, "genes:", len(gene_df))
    print(gene_df.head())

tss_df = parse_gene_tss_from_gtf(GTF_GZ)
print("Loaded gene TSS:", tss_df.shape)

build_atac_gene_table(CDC1_DIFFBIND, OUT_CDC1, tss_df)
build_atac_gene_table(CDC2_DIFFBIND, OUT_CDC2, tss_df)

Loaded gene TSS: (78334, 3)
Wrote: /projectnb/bf528/students/shrjain/final-project-template-shriyajain22/refs/cdc1_atac_DAR_with_genes.tsv genes: 34926
                       gene  atac_log2fc  atac_fdr
51290  ENSMUSG00000111366.3     4.347133       1.0
51485  ENSMUSG00000123256.1     3.262015       1.0
52233  ENSMUSG00000126107.1    -2.922479       1.0
53597  ENSMUSG00000078537.4    -2.442184       1.0
51294  ENSMUSG00000116799.3    -2.427679       1.0
Wrote: /projectnb/bf528/students/shrjain/final-project-template-shriyajain22/refs/cdc2_atac_DAR_with_genes.tsv genes: 32841
                      gene  atac_log2fc  atac_fdr
473   ENSMUSG00000085450.3     3.286641  0.999988
1423  ENSMUSG00000082513.2    -3.021078  0.999988
47    ENSMUSG00000138745.1    -2.893820  0.999988
203   ENSMUSG00000089957.4    -2.799643  0.999988
3153  ENSMUSG00000128953.1    -2.536426  0.999988
